In [ ]:
import json

import numpy as np
import scipy.io as sio

file_name = "fields_002000.mat"    # or a .json/.yaml SCFT result

# Read an SCFT result (.mat, .json or .yaml) or an L-FTS or CL-FTS fields file (.mat)
if file_name.endswith(".json"):
    with open(file_name) as f:
        data = json.load(f)
elif file_name.endswith((".yaml", ".yml")):
    import yaml
    with open(file_name) as f:
        data = yaml.safe_load(f)
else:
    data = sio.loadmat(file_name, simplify_cells=True)     # MATLAB structures as dicts
nx = np.atleast_1d(np.asarray(data['nx'], dtype=int))
lx = np.atleast_1d(np.asarray(data['lx'], dtype=float))

# The angles of the cell in degrees: saved in the file, or in the checkpoint settings or the input
# of older files ([gamma] in 2D)
angles = data.get('angles')
if angles is None and 'checkpoint_settings_json' in data:
    angles = json.loads(str(data['checkpoint_settings_json']))['angles']
if angles is None:
    angles = data.get('initial_params', {}).get('angles')
angles = np.atleast_1d(np.asarray([] if angles is None else angles, dtype=float))
if angles.size == 0:
    print("The file holds no lattice angles: the cell is drawn with right angles.")
    angles = np.full(3, 90.0)
elif angles.size == 1:
    angles = np.array([90.0, 90.0, angles[0]])

# The grid points sit at (i + offset)/n along each lattice vector, as in the solvers: the cell
# centres, or the nodes along a and b for a space group on hexagonal axes. An SCFT result with a
# space group saves its Hall number: the trigonal and hexagonal settings 430-488 use hexagonal
# axes, except the R settings on rhombohedral axes (SpaceGroup.uses_hexagonal_axes)
hexagonal = False
if 'space_group_hall_number' in data:
    hall = int(data['space_group_hall_number'])
    hexagonal = 430 <= hall <= 488 and hall not in (434, 437, 445, 451, 453, 459, 461)
offsets = np.full(len(nx), 0.5)
if hexagonal:
    offsets[:2] = 0.0

# phi_<type>, the real part phi_<type>_real of CL-FTS, or the phi structure of older files
older = data.get('phi', {})
if 'monomer_types' in data:
    names = [str(name).strip() for name in np.atleast_1d(data['monomer_types'])]
elif older:
    names = list(older)
else:
    names = [key[4:].removesuffix('_real') for key in data if key.startswith('phi_') and not key.endswith('_imag')]
phi = {}
for name in names:
    value = data.get('phi_' + name, data.get('phi_' + name + '_real', older.get(name)))
    if value is None:
        raise KeyError(f"The file holds no concentration of monomer type {name}")
    phi[name] = np.asarray(value).reshape(nx)

In [ ]:
import plotly.graph_objects as go
from scipy.ndimage import map_coordinates

monomer_type = None     # the monomer type to draw; None: the first type of the file

if monomer_type is None:
    monomer_type = next(iter(phi))
values = phi[monomer_type]
alpha, beta, gamma = np.deg2rad(angles)

# Lattice vectors as rows (a along x, b in the xy plane, as in ComputationBox)
cx = np.cos(beta)
cy = (np.cos(alpha) - np.cos(beta)*np.cos(gamma))/np.sin(gamma)
L = np.array([lx[0]*np.array([1.0, 0.0, 0.0]),
              lx[1]*np.array([np.cos(gamma), np.sin(gamma), 0.0]),
              lx[2]*np.array([cx, cy, np.sqrt(1.0 - cx**2 - cy**2)])])

# The grid points sit at (i + offset)/n along each lattice vector: the cell centres, or the
# nodes along a and b on hexagonal axes
if np.allclose(angles, 90.0):
    # Orthogonal cell: the grid points themselves
    X, Y, Z = np.meshgrid(*[(np.arange(n) + o)/n*l for n, o, l in zip(nx, offsets, lx)], indexing='ij')
else:
    # Non-orthogonal cell: an isosurface needs points on a grid along x, y and z, so the
    # periodic field is interpolated linearly onto such a grid; it fills the box around
    # the cell, and the outline below marks the cell. The spacing is the finest distance
    # between the planes of the grid, widened when the box holds over 8 times the points
    corners = np.array(list(np.ndindex(2, 2, 2))) @ L
    low, high = corners.min(0), corners.max(0)
    h = np.min(1.0/(nx*np.linalg.norm(np.linalg.inv(L), axis=0)))
    n_box = np.prod((high - low)/h + 1)
    if n_box > 8*np.prod(nx):
        h *= (n_box/(8*np.prod(nx)))**(1/3)
        print(f"The box around the cell is large: drawn with spacing {h:.3g}.")
    X, Y, Z = np.meshgrid(*[np.arange(lo, hi + h/2, h) for lo, hi in zip(low, high)], indexing='ij')
    fractions = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1) @ np.linalg.inv(L)
    values = map_coordinates(values, (fractions*nx - offsets).T, order=1, mode='grid-wrap').reshape(X.shape)

# The edges of the cell
edges = []
for corner in np.ndindex(2, 2, 2):
    for d in range(3):
        if corner[d] == 0:
            end = list(corner)
            end[d] = 1
            edges += [np.array(corner) @ L, np.array(end) @ L, [None]*3]
edges = np.array(edges, dtype=object)

fig = go.Figure(data=[go.Isosurface(
    x=X.flatten(),
    y=Y.flatten(),
    z=Z.flatten(),
    value=values.flatten(),
    # isomax=1.0,
    # opacity=1.0,
    # surface_count=2, # number of isosurface. 2 by default:only min and max
    isomin=0.0,
    isomax=1.0,
    opacity=0.2,
    surface_count=9, # number of isosurface. 2 by default:only min and max
    #caps=dict(x_show=False, y_show=False, z_show=False),
    colorscale='RdBu', #RdBu, jet, Plotly3, OrRd
    colorbar=dict(thickness=30,tickfont=dict(size=30,color="black")),
    reversescale=True,
    ),
    go.Scatter3d(x=edges[:, 0], y=edges[:, 1], z=edges[:, 2], mode='lines',
                 line=dict(color='black', width=2), showlegend=False)])
fig.update_layout(
    title='phi_' + monomer_type,
    autosize=False,
    width=500,
    height=500,
)
fig.update_scenes(
    aspectmode="data",
    camera_projection_type="orthographic",
    xaxis_visible=False,
    yaxis_visible=False,
    zaxis_visible=False)

# fig.write_html('3d_isosurface.html', auto_open=True)
# fig.write_image('3d_volume.svg')